# A. Introduction

SinoNom Annotation is an end-to-end pipeline for preparing and annotating Sino-Nôm inscription data. It extracts inscription content from PDF documents, detects character regions in page images, and provides a Gradio interface for verifying text, bounding boxes, character conditions, reading order, and crop areas. The resulting structured annotations can be exported as a ZIP file for downstream OCR, restoration, and digital humanities research.

For installation instructions, configuration, and technical details, see the [SinoNom Annotation](https://github.com/VongHaiYen11/SinoNom-Annotation)

# B. Pipeline Execution

## 1. Clone the Repository and Prepare models

Before running the annotation pipeline, clone the project repository and install the required dependencies.

This step will:

* clone the **SinoNom-Annotation** repository from GitHub;
* download or load the required text detection models and model weights;

In [1]:
import os
import kagglehub

# Tải đúng phiên bản model về phiên làm việc
model_path = kagglehub.model_download("nhutminhdang/autohdr-detection-models/other/default/2")

print(f"Đã tải model về thư mục: {model_path}")
print("Các tệp thực tế tải về:")
print(os.listdir(model_path))

Đã tải model về thư mục: /kaggle/input/models/nhutminhdang/autohdr-detection-models/other/default/2
Các tệp thực tế tải về:
['dist']


In [2]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/VongHaiYen11/SinoNom-Annotation.git"
REPO_DIR = Path("/kaggle/working/SinoNom-Annotation")

os.chdir("/kaggle/working")

# Delete existing repo folder
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

# Clone fresh repo
print("Cloning repository...")
subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
    check=True
)

print("Done:", REPO_DIR)

Cloning repository...


Cloning into '/kaggle/working/SinoNom-Annotation'...


Done: /kaggle/working/SinoNom-Annotation


Updating files: 100% (115/115), done.


In [ ]:
from pathlib import Path
import shutil
import os

REPO_DIR = Path("/kaggle/working/SinoNom-Annotation")

# Target directories in repository
CKPT_TARGET_DIR = REPO_DIR / "text_detection/models/ckpts"
DIST_TARGET_DIR = REPO_DIR / "text_detection/models/dists"
CKPT_TARGET_DIR.mkdir(parents=True, exist_ok=True)
DIST_TARGET_DIR.mkdir(parents=True, exist_ok=True)

# Candidate roots to search for model files
candidate_dirs = [
    Path(model_path) if "model_path" in globals() else Path("/kaggle/input/models/nhutminhdang/autohdr-detection-models/other/default/2"),
    Path("/kaggle/input/models/nhutminhdang/autohdr-detection-models/other/default/2"),
    Path("/kaggle/input/models/nhutminhdang/autohdr-detection-models/other/default/1"),
]

damage_config_source = None
damage_model_source = None
dist_source = None

for base in candidate_dirs:
    if not base.exists():
        continue
    # Search for dist
    if (base / "dist").is_dir():
        dist_source = base / "dist"
    elif not dist_source:
        for p in base.rglob("dist"):
            if p.is_dir() and (p / "det_model").exists():
                dist_source = p
                break
    # Search for damage detect files
    if (base / "damage_detect.py").is_file():
        damage_config_source = base / "damage_detect.py"
    elif not damage_config_source:
        for p in base.rglob("damage_detect.py"):
            damage_config_source = p
            break
    if (base / "damage_detect.pth").is_file():
        damage_model_source = base / "damage_detect.pth"
    elif not damage_model_source:
        for p in base.rglob("damage_detect.pth"):
            damage_model_source = p
            break

# Copy dist folder
if dist_source and dist_source.is_dir():
    shutil.copytree(dist_source, DIST_TARGET_DIR, dirs_exist_ok=True)
    ocr_executable = DIST_TARGET_DIR / "det_model" / "det_model"
    if ocr_executable.is_file():
        os.chmod(ocr_executable, os.stat(ocr_executable).st_mode | 0o111)
        print(f"Dist folder ready and det_model marked executable: {ocr_executable}")
    else:
        print(f"Dist folder copied to: {DIST_TARGET_DIR}")
else:
    print("Warning: dist folder not found in candidate paths.")

# Copy damage detection files if available
if damage_config_source and damage_config_source.is_file():
    shutil.copy2(damage_config_source, CKPT_TARGET_DIR / "damage_detect.py")
    print(f"Copied damage config: {damage_config_source}")
else:
    print("Notice: damage_detect.py not in download; pipeline will use OCR & fallback detector.")

if damage_model_source and damage_model_source.is_file():
    shutil.copy2(damage_model_source, CKPT_TARGET_DIR / "damage_detect.pth")
    print(f"Copied damage weights: {damage_model_source}")
else:
    print("Notice: damage_detect.pth not in download; pipeline will use OCR & fallback detector.")

print(f"Checkpoint target directory: {CKPT_TARGET_DIR}")
print(f"Dist target directory: {DIST_TARGET_DIR}")


## 2. Upload the PDF and its image folder

Before running the remaining cells, add a dataset/ folder containing:

- the source `.json` file;
- a **flat folder** of page/inscription images used by the annotation app.

The image filename without its extension must match the `ky_hieu` produced by text extraction. After attaching the dataset, edit `SOURCE_JSON`, and `IMAGE_DIR` in the next cell. 


In [ ]:
from pathlib import Path

REPO_DIR = Path('/kaggle/working/SinoNom-Annotation')

IMAGE_DIR = Path('/kaggle/input/datasets/kirinvn1/nlp-dataset-24122014/images') 
SOURCE_JSON = Path('/kaggle/input/datasets/kirinvn1/nlp-dataset-24122014/student_036.json')

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png'}
if not IMAGE_DIR.is_dir():
    raise FileNotFoundError(f'Image folder not found: {IMAGE_DIR}')

images = sorted(path for path in IMAGE_DIR.iterdir()
                if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES)
if not images:
    raise ValueError(f'No supported images found directly inside: {IMAGE_DIR}')
if len({path.stem for path in images}) != len(images):
    raise ValueError('Image filenames must have unique stems.')

print('Extracted JSON:', SOURCE_JSON)
print('Image folder:', IMAGE_DIR)
print('Images:', len(images))

In [ ]:
import json
from pathlib import Path

records = json.loads(SOURCE_JSON.read_text(encoding='utf-8'))
if not isinstance(records, list):
    raise ValueError(f'Source JSON must contain a JSON array: {SOURCE_JSON}')

print('Source JSON:', SOURCE_JSON)
print('Records:', len(records))
print('Images:', len(images))

## 3. Install all repository requirements

This installs directly into the current Kaggle Python environment. Kaggle may print dependency-conflict warnings for preinstalled packages that this repository does not use; those warnings are intentionally ignored in this notebook.


In [ ]:
import subprocess
import sys

REPO_PYTHON = sys.executable
subprocess.run(
    [REPO_PYTHON, '-m', 'pip', 'install', '--upgrade',
     '-r', str(REPO_DIR / 'requirements.txt')],
    check=True,
)
print('Repository requirements installed into:', REPO_PYTHON)

## 4. Launch the Gradio annotation app

The app receives the uploaded image folder and the JSON produced by the previous cell. Per-image saves and Save-all data are written under `/kaggle/working/annotations`. Stop this cell to shut down Gradio.

In [ ]:
import os
import subprocess
from pathlib import Path

ANNOTATION_DIR = Path('/kaggle/working/annotations')
ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

# Launch the Gradio app with Cloudflare Quick Tunnel (no account required, highly reliable)
command = [
    str(REPO_PYTHON), '-u', str(REPO_DIR / 'gradio/app.py'),
    '--image-dir', str(IMAGE_DIR.resolve()),
    '--source-json', str(SOURCE_JSON.resolve()),
    '--output-dir', str(ANNOTATION_DIR.resolve()),
    '--server-name', '0.0.0.0',
    '--port', '7830',
    '--tunnel', 'cloudflare',
    '--show-detection-logs',
]

environment = dict(os.environ)
environment['MPLBACKEND'] = 'Agg'
environment['PYTHONUNBUFFERED'] = '1'

subprocess.run(
    command,
    cwd=REPO_DIR,
    env=environment,
    check=True,
)
